In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms
from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

UCMERCED_DIR = (
    DATA_DIR
    / "ucmerced"
    / "Images"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
)

FIGURE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
)

CONFUSION_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "confusion_matrices"
)

In [3]:
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(DEVICE)

cuda


In [4]:
classes = sorted(
    [
        folder.name
        for folder in UCMERCED_DIR.iterdir()
        if folder.is_dir()
    ]
)

print("Number of Classes:", len(classes))
print()

for cls in classes:
    print(cls)

Number of Classes: 21

agricultural
airplane
baseballdiamond
beach
buildings
chaparral
denseresidential
forest
freeway
golfcourse
harbor
intersection
mediumresidential
mobilehomepark
overpass
parkinglot
river
runway
sparseresidential
storagetanks
tenniscourt


In [5]:
eurosat_label_map = {
    "AnnualCrop":0,
    "Forest":1,
    "HerbaceousVegetation":2,
    "Highway":3,
    "Industrial":4,
    "Pasture":5,
    "PermanentCrop":6,
    "Residential":7,
    "River":8,
    "SeaLake":9
}

In [6]:
uc_to_eurosat = {

    "agricultural":"AnnualCrop",

    "forest":"Forest",

    "freeway":"Highway",
    "intersection":"Highway",
    "overpass":"Highway",

    "buildings":"Industrial",
    "parkinglot":"Industrial",

    "river":"River",

    "denseresidential":"Residential",
    "mediumresidential":"Residential",
    "sparseresidential":"Residential",
    "mobilehomepark":"Residential",

    "harbor":"SeaLake",
    "beach":"SeaLake"

}

In [7]:
records = []

for uc_class in sorted(uc_to_eurosat.keys()):

    folder = (
        UCMERCED_DIR
        / uc_class
    )

    euro_class = uc_to_eurosat[uc_class]

    label = eurosat_label_map[euro_class]

    for img_path in folder.glob("*.tif"):

        records.append(
            {
                "filepath": str(img_path),
                "uc_class": uc_class,
                "eurosat_class": euro_class,
                "label": label
            }
        )

uc_df = pd.DataFrame(records)

print(uc_df.shape)

uc_df.head()

(1400, 4)


,filepath,uc_class,eurosat_class,label
0,c:\Users\ASUS\dev\projects\satellite-project\d...,agricultural,AnnualCrop,0
1,c:\Users\ASUS\dev\projects\satellite-project\d...,agricultural,AnnualCrop,0
2,c:\Users\ASUS\dev\projects\satellite-project\d...,agricultural,AnnualCrop,0
3,c:\Users\ASUS\dev\projects\satellite-project\d...,agricultural,AnnualCrop,0
4,c:\Users\ASUS\dev\projects\satellite-project\d...,agricultural,AnnualCrop,0


In [8]:
uc_df["eurosat_class"].value_counts()

eurosat_class
Residential    400
Highway        300
SeaLake        200
Industrial     200
AnnualCrop     100
Forest         100
River          100
Name: count, dtype: int64

In [9]:
csv_path = (
    DATA_DIR
    / "processed"
    / "ucmerced_mapped.csv"
)

uc_df.to_csv(
    csv_path,
    index=False
)

print(csv_path)

c:\Users\ASUS\dev\projects\satellite-project\data\processed\ucmerced_mapped.csv


In [10]:
eval_transform = transforms.Compose([

    transforms.Resize(
        (224,224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [11]:
class UCMercedDataset(Dataset):

    def __init__(
        self,
        csv_file,
        transform=None
    ):

        self.df = pd.read_csv(csv_file)

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(
            row["filepath"]
        ).convert("RGB")

        if self.transform:

            image = self.transform(image)

        label = int(row["label"])

        return image, label

In [12]:
uc_dataset = UCMercedDataset(
    csv_path,
    transform=eval_transform
)

print(len(uc_dataset))

1400


In [13]:
uc_loader = DataLoader(
    uc_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
)

print(len(uc_loader))

22


In [14]:
print(UCMERCED_DIR)

c:\Users\ASUS\dev\projects\satellite-project\data\ucmerced\Images


In [15]:
images, labels = next(
    iter(uc_loader)
)

print(images.shape)
print(labels.shape)
print(images.dtype)

torch.Size([64, 3, 224, 224])
torch.Size([64])
torch.float32


In [16]:
def evaluate_model(
    model,
    checkpoint_path,
    loader,
    device,
    model_name
):

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            map_location=device
        )
    )

    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            preds = outputs.argmax(dim=1)

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    report = classification_report(
        all_labels,
        all_preds,
        target_names=[
            "AnnualCrop",
            "Forest",
            "HerbaceousVegetation",
            "Highway",
            "Industrial",
            "Pasture",
            "PermanentCrop",
            "Residential",
            "River",
            "SeaLake"
        ],
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).transpose()

    report_df.to_csv(
        FIGURE_DIR /
        f"{model_name}_ucmerced_report.csv"
    )

    cm = confusion_matrix(
        all_labels,
        all_preds
    )

    plt.figure(figsize=(10,8))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[
            "AnnualCrop",
            "Forest",
            "HerbaceousVegetation",
            "Highway",
            "Industrial",
            "Pasture",
            "PermanentCrop",
            "Residential",
            "River",
            "SeaLake"
        ],
        yticklabels=[
            "AnnualCrop",
            "Forest",
            "HerbaceousVegetation",
            "Highway",
            "Industrial",
            "Pasture",
            "PermanentCrop",
            "Residential",
            "River",
            "SeaLake"
        ]
    )

    plt.title(
        f"{model_name} on UC Merced"
    )

    plt.tight_layout()

    plt.savefig(
        CONFUSION_DIR /
        f"{model_name}_ucmerced_confusion_matrix.png",
        dpi=300
    )

    plt.close()

    metrics = pd.DataFrame(
        {
            "Metric":[
                "Accuracy",
                "Macro_F1"
            ],
            "Value":[
                accuracy,
                macro_f1
            ]
        }
    )

    metrics.to_csv(
        FIGURE_DIR /
        f"{model_name}_ucmerced_metrics.csv",
        index=False
    )

    print(f"{model_name}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Macro F1 : {macro_f1:.4f}")

    return accuracy, macro_f1, report_df

In [17]:
class BaselineCNN(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)

        )

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),

            nn.Flatten(),

            nn.Linear(
                128,
                num_classes
            )

        )

    def forward(self,x):

        x = self.features(x)

        x = self.classifier(x)

        return x

In [18]:
baseline_model = BaselineCNN()

baseline_acc, baseline_f1, baseline_report = evaluate_model(

    model=baseline_model,

    checkpoint_path=
    CHECKPOINT_DIR /
    "baseline_cnn_best.pt",

    loader=uc_loader,

    device=DEVICE,

    model_name="baseline_cnn"

)

baseline_cnn
Accuracy : 0.1786
Macro F1 : 0.0927


In [19]:
resnet18_frozen = models.resnet18(
    weights=None
)

resnet18_frozen.fc = nn.Linear(
    resnet18_frozen.fc.in_features,
    10
)

In [20]:
frozen_acc, frozen_f1, frozen_report = evaluate_model(

    model=resnet18_frozen,

    checkpoint_path=
    CHECKPOINT_DIR /
    "resnet18_frozen_best.pt",

    loader=uc_loader,

    device=DEVICE,

    model_name="resnet18_frozen"

)

resnet18_frozen
Accuracy : 0.3093
Macro F1 : 0.2096


In [21]:
resnet18_ft = models.resnet18(
    weights=None
)

resnet18_ft.fc = nn.Linear(
    resnet18_ft.fc.in_features,
    10
)

In [22]:
finetuned_acc, finetuned_f1, finetuned_report = evaluate_model(

    model=resnet18_ft,

    checkpoint_path=
    CHECKPOINT_DIR /
    "resnet18_finetuned_best.pt",

    loader=uc_loader,

    device=DEVICE,

    model_name="resnet18_finetuned"

)

resnet18_finetuned
Accuracy : 0.4429
Macro F1 : 0.3350


In [23]:
comparison = pd.DataFrame(

    {

        "Model":[

            "Baseline CNN",

            "ResNet18 Frozen",

            "ResNet18 Fine-Tuned"

        ],

        "Accuracy":[

            baseline_acc,

            frozen_acc,

            finetuned_acc

        ],

        "Macro_F1":[

            baseline_f1,

            frozen_f1,

            finetuned_f1

        ]

    }

)

comparison

,Model,Accuracy,Macro_F1
0,Baseline CNN,0.178571,0.092658
1,ResNet18 Frozen,0.309286,0.209564
2,ResNet18 Fine-Tuned,0.442857,0.334958


In [24]:
comparison.to_csv(

    FIGURE_DIR /
    "ucmerced_model_comparison.csv",

    index=False

)

print(

    FIGURE_DIR /
    "ucmerced_model_comparison.csv"

)

c:\Users\ASUS\dev\projects\satellite-project\outputs\figures\ucmerced_model_comparison.csv
